# 3 What does the algorithm know?

In lesson 1 (`04_search`) you handed a search space to Optuna's TPE and read what came back;
in lesson 2 your own study (`make study`) drew every trial at random, because that is what
`hpo` does when you give it no sampler. Both times the sampler was a tool used without
looking inside. This notebook looks inside. Optuna calls the part of a search that picks the
next trial the **sampler**: it gets the space and the trials so far, and returns the values
the next trial will train with. Samplers differ in one thing only, what they *know* when
they pick.

At the end of this notebook you can say, for four samplers, what each one knows, which
shortcoming of the one before it removes, and which one you would pick for a given space
and budget, with a reason taken from this notebook's own runs:

| sampler | what it knows when it picks a trial |
|---|---|
| `GridSampler` | a list of values per axis, nothing about the trials so far |
| `RandomSampler` | the distribution of every axis, nothing about the trials so far |
| `TPESampler` | which combinations of values the best trials so far had, and which the rest had |
| `NSGAIISampler` | a population: the best few trials so far, which it recombines |

You also learn to read a search by the shape of its best-so-far curve, to ask whether the
differences between searches are larger than the noise of a single run, and to report the
number of a winner from data the search never saw. That last one is the number your class
problem asks for.

The data is DTD (describable textures) from lesson 1, on cached vectors of three encoders,
so a trial costs a fraction of a second. That is what makes it affordable to run every
sampler several times and look at the spread.

In [ ]:
import sys
import time

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from huggingface_hub.utils import disable_progress_bars
from loguru import logger
from notebooktester import param
from scipy.special import logsumexp
from scipy.stats import truncnorm
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

from hpo.measure import Scalars
from hpo.record import RunRecord, Store, StudyLog
from hpo.search import Choice, Float, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
disable_progress_bars()  # the download bars of the vector caches
optuna.logging.set_verbosity(optuna.logging.WARNING)

BUDGET = param(40, test=3)  # trials per search
SEARCH_EPOCHS = param(5, test=1)  # epochs per trial
SEEDS = param(tuple(range(6)), test=(0, 1))  # one search per sampler per seed
# seeds no search used: for every rerun of a configuration the search already scored
FRESH_SEEDS = param(tuple(range(100, 106)), test=(100, 101))
FINAL_EPOCHS = param(30, test=2)  # the longest retrain of the winner in 3.6
BATCH, N_CLASSES = 128, 47
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

SPLITS = {}
for enc in ENCODERS:
    cache = VectorCache.from_hub(f"pttrn-io/dtd-{enc}")
    # the cache is stored grouped by label: shuffle before slicing, the same order for
    # every encoder, so row i is the same image in all three
    d = cache.dataset.shuffle(seed=0).with_format("numpy")
    X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
    y = torch.tensor(np.asarray(d["label"]))
    a, b = int(0.60 * len(X)), int(0.80 * len(X))
    SPLITS[enc] = {
        "train": (X[:a], y[:a]),
        "val": (X[a:b], y[a:b]),
        "test": (X[b:], y[b:]),
    }

print({split: len(xy[0]) for split, xy in SPLITS["dinov2-small"].items()})
print(
    {enc: SPLITS[enc]["train"][0].shape[1] for enc in ENCODERS},
    "dimensions per encoder",
)

**A three-way split, and here it matters more than anywhere else.** A search tries dozens
of configurations and keeps the best one. The winner's score is therefore the *maximum* of
dozens of noisy numbers, and a maximum of noisy numbers lies above what the same
configuration scores on a fresh run. The search chooses on **validation**; the winner is
scored on **test**, once, at the end (§3.6). The three splits come from the same shuffled
3,760 rows of the DTD cache, in the same order for every encoder.

### One trial: a head on cached vectors

Every trial trains the head of lesson 1 on the vectors of one encoder: `NeuralNet` with
`depth` hidden layers of `width` units (GELU between them) and 47 class scores out, Adam
with learning rate `lr`, batches of 128. `train_head` returns the accuracy after every
epoch, so a trial can report its whole validation curve to the study, as lesson 1's search
did.

Every trial trains **5 epochs**. That is short on purpose: a search is a *ranking*, and a
short trial makes many of them affordable. Equal epochs give every candidate the same
compute, but not necessarily the ranking it would have at full length: a configuration
that learns fast (a high learning rate, a shallow head) looks better after 5 epochs than
one that gets there slowly. Keep that in mind when you read which configurations won. For
a number you **report** the length has to be justified, and §3.6 measures how much the
length changes the winner's score.

In [ ]:
def train_head(
    config: dict, seed: int, epochs: int, on=("val",)
) -> dict[str, list[float]]:
    """Train the head `config` describes for `epochs` epochs; return its accuracy after
    every epoch on each split in `on`."""
    Xtr, ytr = SPLITS[config["encoder"]]["train"]
    torch.manual_seed(seed)
    model = NeuralNet([Xtr.shape[1], *[config["width"]] * config["depth"]], N_CLASSES)
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()
    accuracy = {split: [] for split in on}
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr) - 1, BATCH):
            idx = perm[i : i + BATCH]
            opt.zero_grad()
            loss_fn(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            for split in on:
                Xe, ye = SPLITS[config["encoder"]][split]
                accuracy[split].append(
                    float((model(Xe).argmax(1) == ye).float().mean())
                )
    return accuracy

## 3.1 Grid: list the values, try the combinations

The sampler that knows least. You give it a list of values per axis, the **lattice**, and
it tries combinations of them. It never looks at a score: trial 30 is chosen exactly as
trial 1 was. Optuna's `GridSampler` walks the combinations in a shuffled order (the seed
sets the order), so a budget smaller than the grid gets a random subset of the lattice.

Below are the two spaces of this notebook, written as `hpo.search.Space` as in lesson 1, and
the grid's lattice for each. Round 1 keeps the encoder fixed to `dinov2-small` and searches
the head; round 2 puts the encoder in the space as well.

**Before you run it:** how many lattice points does each space have, and what part of them
does a budget of 40 trials reach?

In [ ]:
WIDTHS, DEPTHS = [64, 128, 256, 512, 1024], [1, 2, 3]

SPACE = {
    "head": Space(
        {
            "encoder": "dinov2-small",  # fixed: every trial gets it
            "width": Choice(WIDTHS),
            "depth": Choice(DEPTHS),
            "lr": Float(1e-4, 1e-2, log=True),
            "epochs": SEARCH_EPOCHS,
        }
    ),
    "encoder": Space(
        {
            "encoder": Choice(ENCODERS),
            "width": Choice(WIDTHS),
            "depth": Choice(DEPTHS),
            "lr": Float(1e-4, 1e-2, log=True),
            "epochs": SEARCH_EPOCHS,
        }
    ),
}

# the grid's lattice: the values it may draw on every axis the space searches
LATTICE = {
    "encoder": ENCODERS,
    "width": WIDTHS,
    "depth": DEPTHS,
    "lr": [1e-4, 1e-3, 1e-2],
}

for name, space in SPACE.items():
    lattice = {axis: LATTICE[axis] for axis in space.searched()}
    n = int(np.prod([len(values) for values in lattice.values()]))
    print(f"space '{name}': {space.line()}")
    print(f"  grid lattice: {lattice}")
    print(
        f"  {n} points; a budget of {BUDGET} trials visits {min(BUDGET / n, 1):.0%} of them\n"
    )

Grid's cost is the **product** of the axes: one extra axis with three values multiplies the
lattice by three, while the budget stays what it is. That is the shortcoming the next
sampler removes.

Note also what grid *can* draw: three learning rates, `1e-4`, `1e-3` and `1e-2`. If the best
learning rate is `3e-3`, grid cannot find it, however long it runs. The other three samplers
draw `lr` anywhere on the log scale between `1e-4` and `1e-2`.

## 3.2 Random: draw instead of list

`RandomSampler` knows the distribution of every axis and nothing else: every trial draws
every axis afresh, independently of all trials before it. Bergstra and Bengio (2012) gave
the argument that made it the default over grid: when only some axes matter, a grid
re-tests the same few values of the axis that matters at every value of the axes that do
not, while random gives every trial a new value of *every* axis. The picture below draws the
argument for nine trials on two axes, of which only the horizontal one moves the score.

What random does not know is what it has seen. After thirty trials in which every `lr`
above `5e-3` scored badly, it still draws there as often as anywhere else.

In [ ]:
# nine trials on two axes: the score depends on x only, y does nothing
x_axis = np.linspace(0, 1, 200)
score = np.exp(-(((x_axis - 0.72) / 0.08) ** 2))  # one narrow good region on x
grid_x, grid_y = np.meshgrid([1 / 6, 3 / 6, 5 / 6], [1 / 6, 3 / 6, 5 / 6])
rand_x, rand_y = np.random.default_rng(3).uniform(0, 1, (2, 9))

fig, axes = plt.subplots(
    2, 2, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [1, 2.2]}
)
for col, (px, py, name) in enumerate(
    [(grid_x.ravel(), grid_y.ravel(), "grid"), (rand_x, rand_y, "random")]
):
    top, plane = axes[0, col], axes[1, col]
    top.plot(x_axis, score, color="0.4")
    top.plot(px, np.exp(-(((px - 0.72) / 0.08) ** 2)), "o", color="tab:blue")
    top.set_ylabel("score")
    top.set_title(
        f"{name}: 9 trials, {len(np.unique(np.round(px, 6)))} values of x tried"
    )
    plane.plot(px, py, "o", color="tab:blue", ms=8)
    for xi in px:
        plane.axvline(xi, color="tab:blue", lw=0.6, alpha=0.4)
    plane.axvspan(0.6, 0.84, color="tab:green", alpha=0.12)
    plane.set_xlabel("x: the axis that matters")
    plane.set_xlim(0, 1)
    plane.set_ylim(0, 1)
axes[1, 0].set_ylabel("y: an axis that does not")
fig.suptitle(
    "the same budget: grid tests 3 values of the axis that matters, random tests 9"
)
plt.tight_layout()
plt.show()

Read the lower panels downwards: every vertical line is a value of `x` that a trial tested.
Grid's nine trials stand on three lines, so six of them re-test an `x` already tried, at a
`y` that changes nothing; its best trial is on the edge of the good band (green). Random's
nine trials stand on nine lines, and in this draw two fall inside the band. Random can miss
too, with another draw: the band runs from 0.6 to 0.84, a share 0.24 of `x`, so all nine
draws land outside it with chance $0.76^9$, about 8%. What it never does is spend trials on
an `x` it has tried before. In
this notebook `lr` plays the part of `x`: forty trials of random give forty different
learning rates, forty trials of grid give three.

## 3.3 TPE: a model of where the good trials were

`TPESampler` (Tree-structured Parzen Estimator) is the one you used in lesson 1. It removes
random's blindness: before every trial it builds a model of the trials so far, in four
steps.

1. **Split.** Sort the trials so far by score. The best tenth, rounded up, is the **good**
   group, everything else the **rest**. After 10 trials the good group is 1 trial, after 25
   it is 3, after 40 it is 4.
2. **Model each group.** A **Parzen estimator** (also called a kernel density estimate)
   turns a handful of points into a smooth distribution, a **density**: a curve that is high
   where values are likely and low where they are not, so draws from it land mostly where
   it is high. It is built by putting one small bump, a **kernel**, on every point and
   adding the bumps up. (The same word, another thing: lesson 2's kernel was the few weights
   of a convolution; here a kernel is one bump of a density.) TPE puts one kernel on every trial of a group. On
   `lr`, on its log scale, the kernel is a bell centred on the trial's value and as wide as
   the larger gap to its neighbouring trials, so a lone trial gets a wide bell and crowded
   trials get narrow ones. On a categorical axis such as `depth`, the kernel puts most of its
   weight on the trial's own value and a little on every other value. One extra kernel,
   wide over the whole space, keeps every value possible. The model of the good group is
   called $l(x)$, the model of the rest $g(x)$.
3. **Candidates.** Draw 24 candidates from $l$. A candidate is a whole configuration, every
   axis at once.
4. **Pick.** Score every candidate by $l(x)/g(x)$: how much more typical it is of the good
   trials than of the rest. Run the one with the highest score.

In Optuna 5 one kernel covers **all axes together** (`multivariate=True` is the default for
a search with one objective). The kernel of a good trial says "`lr` near `7e-4` *together
with* depth 1 *and* width 512", so TPE knows which *combinations* of values the good trials
had. With `multivariate=False` it models every axis on its own: it would know that good
trials often had depth 1 and often had `lr` near `7e-4`, but not whether the two came
together. ("Tree-structured" is about spaces where an axis exists only for some values of
another, a second layer's width only when depth is 2 or more; the spaces here have none.)

Its first 10 trials are random (`n_startup_trials=10`): it needs trials before it has
anything to model. With the same seed they are the very trials `RandomSampler` draws, so
TPE and random part ways only from trial 11 on. §3.5 draws the model of one real TPE search
after 25 trials, with its candidates and its pick.

Where it fails: it can only model what the trials so far show. With 40 trials and 10 of them
random, it has 30 trials to learn from and a good group of at most 4 trials. When the
scores of good configurations differ by less than the noise of one run, which trials end up
in the good group is partly luck.

## 3.4 A population: keep the best and recombine them

`NSGAIISampler` (NSGA-II, a genetic algorithm) knows something else again. It keeps a
**population**: the best `population_size` trials so far, here 8. It works in
**generations** of 8 trials. The first generation is random. Every next trial is a
**child** made from the population in three steps:

1. **Parents.** Pick two members of the population at random and keep the better one; do
   that again for the second parent, among the members other than the first. Better
   trials become parents more often.
2. **Crossover.** Per axis, the child takes the value of one parent or the other, each with
   chance one half. (One child in ten skips this and is a copy of a single member, picked
   at random from the population.) Crossover only passes values on: a child's `lr` is
   always one of its parents' `lr` values, never one in between.
3. **Mutation.** Every axis is then drawn afresh with chance one over the number of axes,
   1/3 in round 1 (width, depth, lr) and 1/4 in round 2 (with the encoder). So on average
   one axis per child is new, and that is the only way values that no parent has come in.

When a generation is done, the population becomes the best 8 of the old population and the
new children.

A child can also come out identical to a trial already run: a copy whose axes all escape
mutation, or two parents that agree on every axis. In this notebook every trial of a search
trains with the same torch seed, so such a repeat returns exactly the same score: a trial
spent, nothing learned. §3.5 counts these repeats.

Its first generation *is* a random search: with the same seed, Optuna's `NSGAIISampler`
draws its first 8 trials exactly as `RandomSampler` does, so the two part ways only from
trial 9 on.

**Which shortcoming of TPE it removes.** TPE keeps the trials so far only through its model:
it remembers where good and bad trials were as two densities and draws new candidates from
them, so everything it does rests on that model being right. The population builds no model
of the space at all. It keeps the good candidates *themselves* and recombines them, so a good
combination of values is passed on as it is, whatever a model would make of it. What that
costs: it learns only once per generation, because the parents of a generation are fixed
while it runs (a budget of 40 is five generations of 8), and it can repeat trials. Where its
lack of a model would pay, a space in which a model of good and bad regions misleads, is not
run in this notebook. Nor are several objectives at once, its best-known use (that is the
"non-dominated sorting" in the name; notebook 04 explains it, with a front over score and
parameters); in Optuna 5 TPE can search several objectives too, so that is not what sets
the population apart. With one objective, "better" is simply "higher validation accuracy".
This notebook does not show it winning on one objective; exercise E2 gives it a budget of
many generations, where it can.

The four samplers, as Optuna objects. Every search gets its own seed (the sampler's seed and
the torch seed of every trial in it), and every sampler gets the same seeds, the same
space and the same budget. `search` writes each search to its own study,
`lesson3-algorithm-<round>-<sampler>-s<seed>`, through `StudyLog.search`, and every trial
reports its validation accuracy per epoch with `trial.report` and is recorded with
`StudyLog.record`, as in lesson 1. `make optuna-dashboard` shows them all.

In [ ]:
SAMPLERS = {
    "grid": lambda space, seed: optuna.samplers.GridSampler(
        {axis: LATTICE[axis] for axis in space.searched()}, seed=seed
    ),
    "random": lambda space, seed: optuna.samplers.RandomSampler(seed=seed),
    # Optuna 5's defaults: 10 random startup trials, 24 candidates, multivariate kernels
    "tpe": lambda space, seed: optuna.samplers.TPESampler(seed=seed),
    "population": lambda space, seed: optuna.samplers.NSGAIISampler(
        population_size=8, seed=seed
    ),
}


def search(round_name: str, sampler: str, seed: int, budget: int = BUDGET) -> StudyLog:
    """One search: `budget` trials of `sampler` over the space of `round_name`, every
    trial trained with torch seed `seed`, written to its own study."""
    space = SPACE[round_name]
    name = f"lesson3-algorithm-{round_name}-{sampler}-s{seed}"
    # every run of the notebook starts the study afresh: a sampler that found earlier
    # trials in its study would start from them, and the budgets would not be equal
    if name in optuna.get_all_study_names(Store.default().url):
        optuna.delete_study(study_name=name, storage=Store.default().url)
    log = StudyLog.search(
        name=name,
        space=space,
        sampler=SAMPLERS[sampler](space, seed),
        attrs={
            "origin": "03_what_does_the_algorithm_know",
            "sampler": sampler,
            "seed": seed,
        },
    )

    def objective(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        start = time.perf_counter()
        val = train_head(config, seed, config["epochs"])["val"]
        for epoch, accuracy in enumerate(val):
            trial.report(accuracy, epoch)
        log.record(
            trial,
            RunRecord(
                role="trial",
                dataset="dtd",
                config=config,
                fixed=space.fixed(),
                train_seconds=time.perf_counter() - start,
                extra={"sampler": sampler, "seed": seed},
            ),
        )
        return val[-1]

    log.study.optimize(objective, n_trials=budget)
    return log


def race(round_name: str, budget: int = BUDGET) -> dict[str, list[StudyLog]]:
    """Every sampler, once per seed, on the space of `round_name`."""
    start = time.time()
    logs = {
        s: [search(round_name, s, seed, budget) for seed in SEEDS] for s in SAMPLERS
    }
    print(
        f"{len(SAMPLERS)} samplers x {len(SEEDS)} seeds x {budget} trials "
        f"in {time.time() - start:.0f}s"
    )
    return logs


def values(log: StudyLog) -> np.ndarray:
    """The validation accuracy of every trial, in the order the trials ran."""
    return np.array([t.value for t in log.trials(optuna.trial.TrialState.COMPLETE)])


def bests(per_seed: list[StudyLog]) -> np.ndarray:
    """The best validation accuracy of every search, one per seed, in the order of SEEDS."""
    return np.array([values(log).max() for log in per_seed])


def best_table(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """Per sampler: the best validation accuracy of a search, over the seeds."""
    rows = []
    for sampler, per_seed in logs.items():
        best = bests(per_seed)
        rows.append(
            {
                "sampler": sampler,
                "mean": best.mean(),
                "std": best.std(ddof=1),
                "worst seed": best.min(),
                "best seed": best.max(),
            }
        )
    return pd.DataFrame(rows).set_index("sampler")

## 3.5 The race

### Round 1: the head, with the encoder fixed

**Question:** on the head-only space, does it matter which sampler searches? Each sampler
runs 40 trials, once per seed; the number per search is the best validation accuracy it
found.

**Before you run it, write down:** which sampler do you expect to find the best head, and by
how much more than the others? Notebook 02 measured the head's capacity on these same
`dinov2-small` vectors: going from a linear map to a wide head bought a couple of points,
and stacking layers gave them back. Is that a space with much in it for a sampler to find?

In [ ]:
round1 = race("head")
display(
    best_table(round1)
    .style.format("{:.4f}")
    .set_caption(
        f"round 1, head only: best validation accuracy of a search, over {len(SEEDS)} seeds"
    )
)

The table of bests says how high each sampler got; the next table says how it got there. Per
sampler, over its six searches: how many different learning rates it tried, how many of its
trials **repeat** an earlier trial of the same search exactly (same values on every axis,
so, with the search's one torch seed, the same score), and how many trials went to each
encoder (in round 1 the encoder is fixed, so all 240 go to `dinov2-small`).

In [ ]:
AXIS_NAMES = ("encoder", "width", "depth", "lr")


def config_of(trial: optuna.trial.FrozenTrial) -> tuple:
    """A trial's values on every axis, comparable between trials."""
    c = trial.user_attrs["config"]
    return (c["encoder"], c["width"], c["depth"], round(c["lr"], 10))


def repeats(log: StudyLog) -> int:
    """How many trials of a search have exactly the values of an earlier trial."""
    configs = [config_of(t) for t in log.trials(optuna.trial.TrialState.COMPLETE)]
    return len(configs) - len(set(configs))


def drawn(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """Per sampler, summed over its seeds: distinct values of lr, repeated trials (and
    the fewest to most per search), the trials per encoder, and the fewest to most
    distinct runs on dinov2-small per search."""
    rows = []
    for sampler, per_seed in logs.items():
        trials = [
            t for log in per_seed for t in log.trials(optuna.trial.TrialState.COMPLETE)
        ]
        encoders = pd.Series([t.user_attrs["config"]["encoder"] for t in trials])
        rep = [repeats(log) for log in per_seed]
        dino = [  # distinct runs on dinov2-small per search: new tickets in the good region
            len(
                {
                    config_of(t)
                    for t in log.trials(optuna.trial.TrialState.COMPLETE)
                    if t.user_attrs["config"]["encoder"] == "dinov2-small"
                }
            )
            for log in per_seed
        ]
        rows.append(
            {
                "sampler": sampler,
                "trials": len(trials),
                "distinct lr": len({round(t.params["lr"], 8) for t in trials}),
                "repeats": sum(rep),
                "repeats per search": f"{min(rep)} to {max(rep)}",
                **{enc: int((encoders == enc).sum()) for enc in ENCODERS},
                "distinct dinov2 runs per search": f"{min(dino)} to {max(dino)}",
            }
        )
    return pd.DataFrame(rows).set_index("sampler")


def winners(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """The winning trial of every search: its validation accuracy and its config."""
    rows = []
    for sampler, per_seed in logs.items():
        for seed, log in zip(SEEDS, per_seed):
            best = log.study.best_trial
            rows.append(
                {
                    "sampler": sampler,
                    "seed": seed,
                    "val": best.value,
                    **{k: best.user_attrs["config"][k] for k in AXIS_NAMES},
                }
            )
    return pd.DataFrame(rows)


_fresh_cache: dict[tuple, float] = {}


def fresh(config: dict) -> float:
    """A configuration's own score: its mean validation accuracy over `FRESH_SEEDS`, after
    the search's epochs. No search used these seeds."""
    key = tuple(config[k] for k in AXIS_NAMES)
    if key not in _fresh_cache:
        _fresh_cache[key] = float(
            np.mean(
                [train_head(config, s, SEARCH_EPOCHS)["val"][-1] for s in FRESH_SEEDS]
            )
        )
    return _fresh_cache[key]


def fresh_scores(logs: dict[str, list[StudyLog]]) -> dict[str, np.ndarray]:
    """Per sampler: the winner of every search, scored on the fresh seeds, in seed order."""
    return {
        sampler: np.array(
            [fresh(log.study.best_trial.user_attrs["config"]) for log in per_seed]
        )
        for sampler, per_seed in logs.items()
    }


display(
    drawn(round1).style.set_caption(
        "round 1: what each sampler drew, all seeds together"
    )
)

**Two measures of a search.** The table above ranks each search by its **best score**: the
highest validation accuracy among its 40 trials. That number is a maximum of 40 noisy runs,
so it lies above what its winning configuration scores on a fresh run (cell 2), and a
sampler that puts more of its trials into the good region draws more tickets in the same
lottery: its maximum comes out higher even when the configurations it finds are no better.
So every search is also scored a second way: its winning configuration is trained again on
the six `FRESH_SEEDS`, which no search used, for the same 5 epochs, and the mean validation
accuracy is the **winner on fresh seeds**. That is the score of the configuration the search
found, not of its luckiest run. The gap between the two is the search's **optimism**.

**Is a difference between two samplers larger than chance?** Both measures move from seed to
seed, so a difference between two samplers means something only when it is large compared
with that movement. Lesson 2 measured it in **standard errors of the difference**, and this
notebook uses the same measure and the same line for every comparison it makes.

Here the runs come in **pairs**. Every sampler runs with the same six seeds, and a seed
fixes the torch seed of every trial in the search (the same configuration scores the same
under every sampler) and the first trials of TPE and the population (they are random's
first trials). So two searches with the same seed share part of their luck, and the fair
comparison subtracts per seed: with $d_s$ the difference between the two samplers under
seed $s$, and $n$ seeds,

$$\text{difference} = \bar d, \qquad \text{standard error} = \frac{\text{std}(d)}{\sqrt n}.$$

The standard error is how much $\bar d$ moves by chance alone. Lesson 2's line reads the
ratio: **within about 2 standard errors, no result; 2 to 3, marginal; beyond 3, a result.**
With six seeds the standard error is itself estimated from six numbers, and this notebook
reads about a dozen such ratios, so a single ratio just over 3 is thin evidence on its own.
The table puts every sampler against random, the sampler that knows nothing about the trials
so far, on both measures.

One more thing before reading numbers: an accuracy on 752 validation images moves in steps
of 1/752, about 0.0013, so the same values (0.7846, 0.7673) recur across searches.

In [ ]:
def paired(a: np.ndarray, b: np.ndarray) -> tuple[float, float, float]:
    """`a` and `b` hold one number per seed, the same seeds in the same order. Returns the
    mean difference a - b, its standard error, and the difference in standard errors."""
    d = np.asarray(a) - np.asarray(b)
    se = float(d.std(ddof=1) / np.sqrt(len(d)))
    return float(d.mean()), se, float(d.mean() / se) if se > 0 else np.nan


def measures(logs: dict[str, list[StudyLog]]) -> dict[str, dict[str, np.ndarray]]:
    """Both measures of every search, per sampler, in seed order."""
    best = {sampler: bests(per_seed) for sampler, per_seed in logs.items()}
    return {"best score": best, "winner on fresh seeds": fresh_scores(logs)}


def compare(a: dict, b: dict, rows: list[tuple[str, str, str]]) -> pd.DataFrame:
    """One line per (label, key in a, key in b): a[key_a] - b[key_b] on both measures,
    paired by seed."""
    out = []
    for label, key_a, key_b in rows:
        line = {"": label}
        for measure in a:
            diff, se, ratio = paired(a[measure][key_a], b[measure][key_b])
            line[(measure, "difference")] = diff
            line[(measure, "standard error")] = se
            line[(measure, "in standard errors")] = ratio
        out.append(line)
    table = pd.DataFrame(out).set_index("")
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    return table


def show(table: pd.DataFrame, caption: str) -> None:
    ratios = [c for c in table.columns if c[1] == "in standard errors"]
    ses = [c for c in table.columns if c[1] == "standard error"]
    display(
        table.style.format("{:+.4f}")
        .format("{:.4f}", subset=ses)
        .format("{:+.1f}", subset=ratios)
        .set_caption(caption)
    )


def versus_random(m: dict) -> pd.DataFrame:
    others = [s for s in SAMPLERS if s != "random"]
    return compare(m, m, [(f"{s} - random", s, "random") for s in others])


m1 = measures(round1)
show(versus_random(m1), "round 1: every sampler against random, paired by seed")

**Why are the samplers so close?** Two explanations give the same table, and they call for
different actions:

- the space has nothing in it: every configuration scores about the same, so there is
  nothing for any sampler to find;
- the space has bad regions and a wide region of good configurations, every sampler reaches
  the good region within its budget, and the configurations the searches end on are about
  equally good.

The next cell tells them apart with two measurements. First, the spread of *all* trials of
round 1, not only the winners: a space with nothing in it has all of them close together.
Second, the winners of all 24 searches on fresh seeds, next to their best scores: how far
apart are the configurations found, and how much of each best score was luck?

In [ ]:
every1 = np.concatenate(
    [values(log) for per_seed in round1.values() for log in per_seed]
)
print(
    f"all {len(every1)} trials of round 1: "
    + ", ".join(
        f"{name} {np.quantile(every1, q):.3f}"
        for name, q in [
            ("min", 0),
            ("10%", 0.1),
            ("median", 0.5),
            ("90%", 0.9),
            ("max", 1),
        ]
    )
)


def optimism_table(m: dict) -> pd.DataFrame:
    """Per sampler: mean best score, mean winner on fresh seeds, and the optimism."""
    rows = []
    for sampler in SAMPLERS:
        best, again = m["best score"][sampler], m["winner on fresh seeds"][sampler]
        rows.append(
            {
                "sampler": sampler,
                "best score": best.mean(),
                "winner on fresh seeds": again.mean(),
                "fresh: lowest search": again.min(),
                "fresh: highest search": again.max(),
                "optimism": (best - again).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("sampler")


display(
    optimism_table(m1)
    .style.format("{:.4f}")
    .set_caption(
        "round 1: the best score of a search against its winner on fresh seeds"
    )
)
fresh1 = np.concatenate(list(m1["winner on fresh seeds"].values()))
print(
    f"the {len(fresh1)} winners on fresh seeds: from {fresh1.min():.4f} to {fresh1.max():.4f}"
)

**On the head, the samplers cannot be told apart, on either measure.** By best score, TPE's
searches average 0.776, random's 0.774, grid's and the population's 0.770. Against random,
paired by seed: TPE +0.8 standard errors, grid −1.7, the population −1.4. On the winners'
fresh-seed scores: TPE +0.4, grid −1.5, the population −1.4. All within 2: by the line, no
result.

The draw table shows how differently they got there: grid tried 3 learning rates in 240
trials, random and TPE 240 different ones, the population 112 (a child takes its learning
rate from a parent unless that axis mutates). And the population **repeated** 61 of its 240
trials exactly, 6 to 13 per search: trials that returned a score it already had. Random,
grid and TPE repeated none.

**The space is not flat.** The trials of round 1 run from 0.10 to 0.785: a tenth of them
score below 0.59, half below 0.739. There are bad regions (you see them in the next
figure), and a search that landed in them would lose many points.

**But the configurations the searches end on lie close together.** On fresh seeds the 24
winners lie between 0.752 and 0.772, and per sampler their means are 0.759 to 0.766, with no
difference the line can see. Every best score sits about a point above its own winner's
fresh-seed score (the optimism column: 0.9 to 1.1 points). A search ends on the
configuration whose *run* was luckiest among many good ones.

So the reading is not "there is nothing to find", but: **every sampler reaches the wide good
region within 40 trials, and inside it the searches cannot pick out a better configuration
than one another.** More search on this space with any sampler buys little.

Where the samplers drew, for one seed. Every dot is a trial: its position is the trial's
number and the learning rate it drew, its colour the validation accuracy, its size the
depth.

In [ ]:
fig, axes = plt.subplots(1, len(SAMPLERS), figsize=(15, 3.6), sharey=True)
for ax, (sampler, per_seed) in zip(axes, round1.items()):
    trials = per_seed[0].trials(optuna.trial.TrialState.COMPLETE)
    points = ax.scatter(
        [t.number + 1 for t in trials],
        [t.params["lr"] for t in trials],
        c=[t.value for t in trials],
        cmap="viridis",
        vmin=0.6,
        vmax=0.8,
        s=[12 * t.params["depth"] ** 2 for t in trials],
    )
    ax.set_yscale("log")
    ax.set_title(f"{sampler}, seed {SEEDS[0]}")
    ax.set_xlabel("trial")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("lr drawn (log axis)")
fig.colorbar(points, ax=axes, label="validation accuracy")
fig.suptitle(
    "round 1: the lr every trial drew, in the order the trials ran (dot size = depth)"
)
plt.show()

**Grid** draws on three horizontal lines, one per learning rate in its lattice, and in no
order you can read: it does not look at the colours. Its darkest dots, the worst trials,
are on the `1e-4` line with the larger depths: at 5 epochs, a low learning rate has not
got far yet. **Random** scatters over the whole log range from the first trial to the last,
the same density everywhere, including the low learning rates around `1e-4` where its
darkest dots are too. **TPE** starts the same way (its first 10 trials are random's first
10), and from about trial 18 its draws gather between about `2e-4` and `2e-3`, mostly with
the smallest depth, where the bright dots are. **The population** shares its first 8 trials
with random (same seed), and from the second generation on its trials sit on a few learning
rates around `1.5e-3`, the values of its best parents, with an occasional mutation off the
line.

That is the difference in what they know, in one picture: grid and random never change
where they draw; TPE and the population move to where the good trials were.

### Inside TPE: one pick, drawn

§3.3 described TPE's four steps in words. The next cell carries them out by hand on the TPE
search of seed 0, after its first 25 trials, with Optuna 5's own rules (the best tenth
rounded up is good, the kernel widths, the extra wide kernel, 24 candidates scored by
$l(x)/g(x)$), so you can see what TPE knew when it chose trial 26. The left panel is the
`lr` axis: the two models $l$ and $g$ as densities on `lr` alone, the trials they were built
from (marks along the bottom), the 24 candidates drawn from $l$ and the one with the highest
score. A candidate sits at the height of $l$ at its `lr`, but its colour is its score over
all three axes, so two candidates at about the same `lr` can differ in colour when their
width or depth differ. The right
panels are the two categorical axes: the weight each model gives every value.

The candidates are a fresh draw from the same model, not the exact 24 TPE drew at that
moment (those depend on its random number generator), so the cell also prints the trial TPE
actually ran next.

**Before you run it:** look at TPE's panel of the figure above, trials 1 to 25. Where on the
`lr` axis are its brightest dots, the trials that will form the good group? Where are most
of the other dots? From that, where do you expect $l$ to be high and $g$ low?

In [ ]:
AXES = {"width": WIDTHS, "depth": DEPTHS}  # the categorical axes of round 1
LOW, HIGH = np.log(1e-4), np.log(1e-2)  # the lr axis, on its log scale


def kernels(group: list) -> dict[str, np.ndarray]:
    """TPE's model of a group of trials, by Optuna 5's rules: one kernel per trial over all
    axes together, plus one wide kernel over the whole space."""
    n = len(group)
    x = np.log([t.params["lr"] for t in group])
    # lr: a bell on the log scale, as wide as the larger gap to its neighbours; the
    # outermost trials take the gap to their inner neighbour; never narrower than
    # the range / (n + 2)
    order = np.argsort(x)
    edges = np.concatenate([[LOW], x[order], [HIGH]])
    gaps = np.maximum(edges[1:-1] - edges[:-2], edges[2:] - edges[1:-1])
    if n >= 2:
        gaps[0], gaps[-1] = edges[2] - edges[1], edges[-2] - edges[-3]
    sigma = np.empty(n)
    sigma[order] = np.clip(gaps, (HIGH - LOW) / min(100, n + 2), HIGH - LOW)
    model = {
        "mu": np.append(x, (LOW + HIGH) / 2),
        "sigma": np.append(sigma, HIGH - LOW),
    }
    # a categorical axis: weight 1 on the trial's own value, 1/(n+1) on every value
    for axis, choices in AXES.items():
        w = np.full((n + 1, len(choices)), 1 / (n + 1))
        for k, t in enumerate(group):
            w[k, choices.index(t.params[axis])] += 1
        model[axis] = w / w.sum(1, keepdims=True)
    return model


def lr_bells(model: dict, lr: np.ndarray) -> np.ndarray:
    """The log density of every kernel's bell at every lr: shape (len(lr), kernels)."""
    mu, sigma = model["mu"], model["sigma"]
    a, b = (LOW - mu) / sigma, (HIGH - mu) / sigma
    return truncnorm.logpdf(np.log(lr)[:, None], a, b, loc=mu, scale=sigma)


def log_density(model: dict, lr, width, depth) -> np.ndarray:
    """log l(x) or log g(x) of whole configurations: per kernel the product over the axes,
    then the mean over the kernels."""
    per_kernel = lr_bells(model, np.asarray(lr))
    for axis, values_ in (("width", width), ("depth", depth)):
        idx = [AXES[axis].index(v) for v in values_]
        per_kernel = per_kernel + np.log(model[axis][:, idx]).T
    return logsumexp(per_kernel, axis=1) - np.log(len(model["mu"]))


def draw(model: dict, n: int, rng: np.random.Generator) -> dict[str, np.ndarray]:
    """n candidates from a model: pick a kernel, then every axis from that kernel."""
    k = rng.integers(len(model["mu"]), size=n)
    mu, sigma = model["mu"][k], model["sigma"][k]
    z = truncnorm.rvs(
        (LOW - mu) / sigma, (HIGH - mu) / sigma, loc=mu, scale=sigma, random_state=rng
    )
    out = {"lr": np.exp(z)}
    for axis, choices in AXES.items():
        out[axis] = np.array(
            [choices[rng.choice(len(choices), p=model[axis][i])] for i in k]
        )
    return out


AT = min(25, BUDGET - 1)
trials = round1["tpe"][0].trials(optuna.trial.TrialState.COMPLETE)
so_far, ran_next = trials[:AT], trials[AT]
n_good = min(int(np.ceil(0.1 * AT)), 25)
ranked = sorted(so_far, key=lambda t: t.value, reverse=True)
good, rest = ranked[:n_good], ranked[n_good:]
l, g = kernels(good), kernels(rest)

candidates = draw(l, 24, np.random.default_rng(0))
score = log_density(l, **candidates) - log_density(g, **candidates)
pick = int(np.argmax(score))

print(f"after {AT} trials: {len(good)} good, {len(rest)} rest")
for t in good:
    print(
        f"  good: trial {t.number + 1:2d}, val {t.value:.4f}, lr {t.params['lr']:.1e}, "
        f"width {t.params['width']}, depth {t.params['depth']}"
    )
print(
    f"the pick of 24 candidates: lr {candidates['lr'][pick]:.1e}, width "
    f"{candidates['width'][pick]}, depth {candidates['depth'][pick]}, "
    f"log l - log g = {score[pick]:.2f}"
)
print(
    f"the trial TPE ran next (trial {ran_next.number + 1}): lr {ran_next.params['lr']:.1e}, "
    f"width {ran_next.params['width']}, depth {ran_next.params['depth']}, "
    f"val {ran_next.value:.4f}"
)

fig, axes = plt.subplots(
    1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [2.4, 1, 1.4]}
)
grid_lr = np.geomspace(1e-4, 1e-2, 400)
for model, label, color in [
    (l, "l: good trials", "tab:green"),
    (g, "g: the rest", "0.5"),
]:
    density = np.exp(
        logsumexp(lr_bells(model, grid_lr), axis=1) - np.log(len(model["mu"]))
    )
    axes[0].plot(grid_lr, density, color=color, label=label)
top = axes[0].get_ylim()[1]
axes[0].plot(
    [t.params["lr"] for t in rest], [-0.04 * top] * len(rest), "|", color="0.5", ms=10
)
axes[0].plot(
    [t.params["lr"] for t in good],
    [-0.04 * top] * len(good),
    "^",
    color="tab:green",
    ms=8,
)
l_at = np.exp(logsumexp(lr_bells(l, candidates["lr"]), axis=1) - np.log(len(l["mu"])))
points = axes[0].scatter(
    candidates["lr"],
    l_at,
    c=score,
    cmap="viridis",
    s=30,
    zorder=3,
    label="24 candidates: height = l on lr alone, colour = score on all axes",
)
axes[0].scatter(
    candidates["lr"][pick],
    l_at[pick],
    marker="*",
    s=300,
    color="crimson",
    zorder=4,
    label="the pick",
)
axes[0].axvline(
    ran_next.params["lr"],
    color="crimson",
    ls=":",
    lw=1,
    label=f"trial {ran_next.number + 1}, as TPE ran it",
)
axes[0].set_xscale("log")
axes[0].set_xlabel("lr (log axis)")
axes[0].set_ylabel("density on the log-lr axis\n(high: draws land here often)")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.18), ncol=2)
fig.colorbar(points, ax=axes[0], label="score: log l(x) - log g(x), all axes")
for ax, axis in zip(axes[1:], ["depth", "width"]):
    pos = np.arange(len(AXES[axis]))
    ax.bar(pos - 0.2, l[axis].mean(0), 0.4, color="tab:green", label="l")
    ax.bar(pos + 0.2, g[axis].mean(0), 0.4, color="0.6", label="g")
    ax.set_xticks(pos, AXES[axis])
    ax.set_xlabel(axis)
    ax.set_ylabel("weight")
axes[1].legend()
fig.suptitle(
    f"TPE, round 1, seed {SEEDS[0]}: its model after {AT} trials, and how it picks trial {AT + 1}"
)
plt.tight_layout()
plt.show()

**What TPE knew after 25 trials.** The good group is trials 20, 21 and 25, all three width
1024 and depth 1, with learning rates between `3.8e-4` and `4.8e-4`.

- **The `lr` panel.** $l$ (green) is one broad hill over roughly `2e-4` to `1e-3`. The three
  good trials sit close together, but Optuna never makes a bell narrower than the log range
  divided by the number of trials plus two (a fifth of it here), and the extra wide kernel
  adds a floor everywhere. $g$ (grey) has its bumps where the other 22 trials went, the
  highest around `1e-3` to `2e-3`. The score $l/g$ is high where the green line is high and
  the grey line low, and low around `1e-3` to `2e-3`, where many trials of the rest sit.
- **The categorical panels.** $l$ gives depth 1 a weight of 0.62, but $g$ gives it 0.55:
  most of the rest had depth 1 too, so depth hardly separates good from rest, and a
  candidate with depth 2 loses little. Width separates more: $l$ gives width 1024 a weight
  of 0.47, $g$ only 0.26. Because each kernel covers all axes together, a
  candidate scores well only when its `lr`, width and depth sit together the way they did in
  a good trial.
- **The pick.** In this draw the best candidate is `lr = 1.3e-4` with width 1024 and depth
  1: $l$ is moderate there, but $g$ is low, because few trials of the rest went that far
  down. The ratio rewards values like the good trials *and* unlike the rest, so it also
  sends TPE to where the rest has not been.

The trial TPE actually ran next came from another draw of 24: `lr = 5.8e-4`, width 1024,
depth 2, and it scored 0.7606, below all three good trials. TPE picks the best of the
candidates it drew; with a good group of three trials and 24 random candidates, a pick can
still miss.

### Round 2: put the encoder in the space

The encoder is an axis that moves the score a lot (lesson 1 measured a gap of more than ten
points between DINOv2 and resnet-18 on DTD). So let the search choose it.

**Question:** does a search get better when an axis that matters is added to its space?

**Before you run it, write down:** for each sampler, will the best score in round 2 be
higher, equal or lower than in round 1, and why?

In [ ]:
round2 = race("encoder")
display(
    drawn(round2).style.set_caption(
        "round 2: what each sampler drew, all seeds together"
    )
)
m2 = measures(round2)
display(
    optimism_table(m2)
    .style.format("{:.4f}")
    .set_caption(
        "round 2: the best score of a search against its winner on fresh seeds"
    )
)
show(versus_random(m2), "round 2: every sampler against random, paired by seed")
print(
    "optimism, TPE minus random, paired by seed (best-score difference minus fresh difference):"
)
for name, m in [("round 1", m1), ("round 2", m2)]:
    gap = {s: m["best score"][s] - m["winner on fresh seeds"][s] for s in SAMPLERS}
    diff, se, ratio = paired(gap["tpe"], gap["random"])
    print(
        f"  {name}: {diff:+.4f}, standard error {se:.4f}, {ratio:+.1f} standard errors"
    )


def near_best(log: StudyLog, pilot: int = 10, points: float = 0.01) -> float:
    """A pilot: the first `pilot` trials of a search, and the share of them within
    `points` of their own best. A high share says the good region is wide."""
    v = values(log)[:pilot]
    return float((v >= v.max() - points).mean())


print("pilot of 10 random trials: share within one point of the pilot's best")
for name, logs in [("round 1", round1), ("round 2", round2)]:
    shares = [near_best(log) for log in logs["random"]]
    print(f"  {name}: {min(shares):.0%} to {max(shares):.0%} per search")

**Where the samplers spend their trials now differs a lot.** Grid and random spread their
trials evenly over the three encoders, 74 to 88 trials each of 240, so about two thirds of
their budget went to `resnet-18` and `mobilenet_v2`. TPE put 160 of its 240 trials on
`dinov2-small`, the population 135. What counts for a search's best score is the number of
*distinct* runs on `dinov2-small`, each a new draw of the seed lottery: 26 to 27 per search for TPE, 10 to 16 for random,
10 to 18 for grid and 11 to 16 for the population. The
population's 135 include its repeats (61 in this round, 6 to 14 per search), which draw no
new ticket.

**By best score, TPE comes out ahead.** TPE's searches average 0.774, grid's and the
population's 0.766, random's 0.763; against random, paired by seed, TPE is +1.1 points at
+3.2 standard errors.

**By the winners on fresh seeds, it does not.** The configurations the searches found score
0.760 to 0.763 per sampler, and against random TPE is +0.1 points at +0.3 standard errors,
grid −0.6, the population +0.3: no result anywhere. "No result" is not "the same": two
standard errors of these differences are about 0.6 points for TPE against random (standard
error 0.0032) and 0.9 points for the population (0.0044), so an advantage of half a point
would not have shown at six seeds. What the comparison rules out is an advantage of more
than about 0.6 points.

**What separates TPE is its optimism**: its best score lies 1.1 points above its winner's
fresh-seed score, random's only 0.1. That is not a second result next to the +3.2: optimism
is best score minus fresh score, so TPE's extra optimism (+3.1 standard errors, printed
under the table above) is the +3.2 on best score minus a fresh-seed difference near zero.
The distinct draws on `dinov2-small` go with it: TPE, with
about twice random's distinct runs there, has the largest optimism (0.011), random the
smallest (0.001). The population put 135 trials on `dinov2-small` but, with its repeats, only
11 to 16 distinct runs per search, about as many as random, and its optimism is low too
(0.003). Grid, with 10 to 18, lies in between (0.006), so the count is not the whole story;
but the best of more draws of the same lottery lies higher, and TPE drew the most.

Two caveats on what "optimism" measures. The fresh-seed scores use the same 752 validation
images that chose the winners, so optimism here is the seed lottery only, not the luck of
which images happened to suit a configuration. That remaining luck favours the sampler that
tried the most distinct configurations, TPE, so if anything it flatters TPE's fresh-seed
score, and the null result against random is conservative.

**The samplers differ in where they spend their trials and in how optimistic their best
score is; in the model they found at this budget, the fresh-seed comparison shows no
difference larger than about 0.6 points.** A sampler that found configurations better by
more than that would have shown it on fresh seeds; none did.

Note grid in the draw table: Optuna's `GridSampler` walks its lattice in a shuffled order, so
a grid cut off at 40 of 135 points still tried all three encoders. A grid written as nested
`for` loops with the encoder outermost has 45 points per encoder, so cut off after 40 it
would have tried one encoder only.

What did the samplers find? First, every trial of round 2 grouped by encoder: is there a
difference between the encoders for a search to find at all? Then, per search, how many
trials went to the two weak encoders after the startup, when TPE and the population use
what they know. Then the winner of every single search.

In [ ]:
trials2 = pd.DataFrame(
    [
        {
            "sampler": sampler,
            "seed": seed,
            "trial": t.number + 1,
            "encoder": t.user_attrs["config"]["encoder"],
            "val": t.value,
        }
        for sampler, per_seed in round2.items()
        for seed, log in zip(SEEDS, per_seed)
        for t in log.trials(optuna.trial.TrialState.COMPLETE)
    ]
)
display(
    trials2.groupby("encoder")["val"]
    .agg(trials="count", worst="min", median="median", best="max")
    .sort_values("best", ascending=False)
    .style.format("{:.4f}", subset=["worst", "median", "best"])
    .set_caption(
        "round 2, every trial of every sampler: validation accuracy per encoder"
    )
)


# after the startup: trials 11 to the end, where TPE models and the population recombines
def lowest_to_highest(frame: pd.DataFrame) -> pd.Series:
    """Per sampler: the fewest and the most rows of `frame` one search has."""
    every_search = pd.MultiIndex.from_product([list(SAMPLERS), SEEDS])
    counts = (
        frame.groupby(["sampler", "seed"]).size().reindex(every_search, fill_value=0)
    )
    return counts.groupby(level=0).agg(lambda s: f"{s.min()} to {s.max()}")


weak = trials2[(trials2["trial"] > 10) & (trials2["encoder"] != "dinov2-small")]
per_search = pd.DataFrame(
    {
        f"of trials 11 to {BUDGET}": lowest_to_highest(weak),
        "of the last 10": lowest_to_highest(weak[weak["trial"] > BUDGET - 10]),
    }
).reindex(list(SAMPLERS))
display(
    per_search.style.set_caption(
        "round 2: trials on resnet-18 or mobilenet_v2 per search, lowest to highest over the seeds"
    )
)

table = winners(round2)
display(
    table.style.format({"val": "{:.4f}", "lr": "{:.1e}"})
    .hide(axis="index")
    .set_caption("round 2: the winner of every search")
)

**The encoder axis has a lot in it.** The best `dinov2-small` trial scores 0.785; the best
`mobilenet_v2` trial 0.636 and the best `resnet-18` trial 0.617, about fifteen points lower.
Even the median `dinov2-small` trial (0.741) is ten points above the best trial of either
other encoder.

**After the startup, what each sampler knows shows in its draws.** Of trials 11 to 40, grid
and random spent 16 to 25 per search on the weak encoders, still 5 to 8 of the last ten.
TPE spent 4 to 8, and 0 to 3 of the last ten: it cut its draws of the weak encoders from
about two thirds to about a fifth, but did not stop, because the score $l/g$ always leaves
some weight on every value. The population lies in between, 7 to 15.

**Every one of the 24 searches picked `dinov2-small`**, and that says something: every
sampler, grid included, drew all three encoders, so each search had the choice and made it.
Twelve `dinov2-small` trials were enough for that. The winners also agree on the head:
depth 1 in 19 of 24, and a learning rate between `1e-4` and `2.5e-3`. (Remember the 5-epoch
trials: a shallow head learns fast, which may be part of why depth 1 wins; this notebook
does not test it.) Five of grid's six winners have `lr = 1e-3`: its lattice happens to hold
a value inside that band, so here its three learning rates did not cost it.

### Did the encoder axis make the search better or worse?

Per sampler, round 2 against round 1, on both measures and with the line of round 1. The
pairs are again the seeds: a sampler with seed $s$ in round 2 against the same sampler with
seed $s$ in round 1.

In [ ]:
show(
    compare(m2, m1, [(s, s, s) for s in SAMPLERS]),
    "round 2 minus round 1, per sampler, paired by seed",
)

**By best score, the encoder axis looks costly for the samplers that do not learn.** Random
lost 1.1 points (−4.0 standard errors), grid 0.4 (−2.0, on the edge of marginal), the
population 0.4 (−1.4) and TPE 0.2 (−0.7). Random's loss has a plain cause: about two thirds
of its trials went to the weak encoders, so on average about 12 of its 40 trials (74 of 240)
went to the encoder that can win, where round 1 had all 40.

**By the winners on fresh seeds, the axis cost nothing measurable.** Random −0.35 points
(−1.2 standard errors), TPE −0.3 (−1.0), grid +0.05 (+0.7), the population +0.3 (+0.6): no
result for any sampler. For random the comparison rules out a loss of more than about 0.6
points (two standard errors of 0.0028 around −0.35): its 12 `dinov2-small` trials found a
configuration not measurably worse than its 40 trials in round 1. What it lost was lottery
tickets, and with them the height of its best score.

So, at this budget and on this plateau, the encoder axis changed **where** the samplers
spent their trials and **how high** their best scores came out, and nothing that the line
can see, at a resolution of about 0.6 points, in the configurations they found. The axis would cost the model too if the good
region were narrow or the budget small, so that 12 trials could miss it; this notebook does
not run that case (exercise E1 does).

> **A search space is a claim about where the answer might be.** A region you already know
> is bad does not make the search more thorough; with a fixed budget it takes trials away
> from the region that matters, unless the sampler learns to leave it. Whether those trials
> were needed is a question for fresh seeds, not for the search's best score.

### Best-so-far: read the shape

A search's **best-so-far curve** is, after every trial, the best score of the trials up to
then. It can only go up. Its *shape* says what more budget would buy, which the final
number does not:

- a curve that is **still climbing** at the last trial says the budget was the limit: more
  trials would likely find more;
- a curve that went **flat** early says the space is exhausted for this sampler: more trials
  of the same search buy nothing, and what would help is another space, not a longer search.

In [ ]:
STYLE = {
    "grid": {"color": "tab:blue"},
    "random": {"color": "tab:orange", "ls": "--"},
    "tpe": {"color": "tab:green"},
    "population": {"color": "tab:red", "ls": ":"},
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
lowest_from_trial_2 = {}
for ax, (logs, title) in zip(
    axes, [(round1, "round 1: head only"), (round2, "round 2: head + encoder")]
):
    for sampler, per_seed in logs.items():
        curves = [np.maximum.accumulate(values(log)) for log in per_seed]
        n = min(len(c) for c in curves)
        curves = np.array([c[:n] for c in curves])
        mean, std = curves.mean(0), curves.std(0)
        x = np.arange(1, n + 1)
        ax.plot(x, mean, label=sampler, **STYLE[sampler])
        lowest_from_trial_2[title] = min(
            lowest_from_trial_2.get(title, 1.0), mean[1:].min()
        )
        ax.fill_between(
            x, mean - std, mean + std, alpha=0.12, color=STYLE[sampler]["color"]
        )
    ax.set_title(title)
    ax.set_xlabel("trials run")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("best validation accuracy so far")
# every line is in view from trial 2 on
for ax, title in zip(axes, lowest_from_trial_2):
    ax.set_ylim(max(lowest_from_trial_2[title] - 0.01, 0.0), 0.79)
axes[0].legend()
fig.suptitle(
    f"best-so-far per sampler: line = mean over {len(SEEDS)} seeds, band = their std"
)
plt.tight_layout()
plt.show()

**Read the start first.** For the first trials the lines of random, TPE and the population
lie on top of each other: with the same seed, TPE's 10 random startup trials and the
population's first generation of 8 are the trials random draws. Where a line leaves random's,
the sampler starts to use what it knows.

**Round 1:** every curve is within about a point of its end after 10 trials, and the last 20
trials add less than a point to any of them (most to random, 0.8 points, least to the
population, almost nothing). TPE is the highest line from trial 16 on, which the race
above did not separate from chance. Grid starts slowest, since its early draws include the `lr=1e-4` and `lr=1e-2`
lines, and reaches its end value by trial 25. A longer search on this space buys little,
whichever sampler runs it.

**Round 2:** the curves start lower, because many early trials land on a weak encoder, and
climb for longer. Grid, the sampler that knows least, is the highest line from trial 2 to
17: about a third of its draws are `lr = 1e-3`, a good value, while random's early draws
spread over the whole range, including the poor learning rates. TPE leaves random after
trial 10 and is the highest line from trial 18 on. Random and the population are still
rising in the last ten trials (the population gains 0.7 points between trial 35 and 40, on
the mean line alone, not tested against chance); grid is nearly flat from trial 28.

What would mean the opposite: a curve still climbing steeply at the last trial says "give
this search more trials"; a curve flat from trial 10 says "give it another space". Round 1
is the second kind; in round 2, random and the population are mildly the first kind, and
TPE, which learned to leave the weak encoders alone, is close to the second.

Keep in mind what these curves measure: the best *score* so far. A curve climbs when a
search finds a better configuration, and also when it simply draws more runs in the good
region, each a new lottery ticket. Round 2's TPE line sits highest for the second reason:
its winners were not distinguishable from random's on fresh seeds. The shape still says what more trials would do to
the search's best score; whether they would find a better model is a question for fresh
seeds.

### Choosing a sampler for a space and a budget

The two rounds give one measured result and a rule of thumb around it. The result: **on a
space whose good region is wide compared with the budget, the sampler changes where the
trials go and how optimistic the best score is; in the configuration found, the fresh-seed
comparison shows no difference larger than about 0.6 points.** Both rounds are that case.

**How wide is the good region? A pilot tells you before you choose.** Run a few random trials
and look at the share that lands within a point of the pilot's own best. A high share says
the good region is wide, so any sampler will reach it and the choice matters little; a low
share says it is narrow, so a sampler that concentrates its draws has something to gain. The
first 10 trials of every random search are such a pilot, printed under the round-2 table:
between 10% and 50% of them lie within a point of the pilot's best in round 1, between 10%
and 30% in round 2, where two thirds of random's draws go to the weak encoders. Ten trials
make a coarse measure (10% is the best trial alone), so read it as a first look, and give
the pilot more trials when the choice depends on it. The quantiles of all 960 round-1 trials
in §3.5 are the same measure taken on every trial: the best tenth lies within 1.6 points of
the best trial (0.769 to 0.785). In both rounds every sampler reached that region; a space
where a pilot of this size finds next to nothing near its best is not run here.

The rule of thumb says when a sampler is the right pick, from what it knows; the last two
columns say what this notebook shows for it and what it leaves to an exercise.

| pick | when | why, from what it knows | shown here | left to an exercise |
|---|---|---|---|---|
| grid | the lattice fits in the budget, and its values are ones you need | it visits every point once and repeats nothing; it cannot find a value off the lattice | round 1: 3 learning rates in 240 trials; round 2: its lattice held `lr = 1e-3`, a good value, and its winners were not distinguishable from random's on fresh seeds | a lattice that fits the budget: E1 (a) |
| random | a small budget, no idea yet which axes matter, or a budget at or below TPE's 10 startup trials | it gives every trial a new value of every axis and assumes nothing; below 10 trials TPE *is* random | both rounds: no other sampler's winners were distinguishable from its winners on fresh seeds, and in round 2 its best score lay only 0.1 points above them; it is the control every other sampler is read against | a budget of 10: E1 (b) |
| TPE | a budget well past its startup, in a space whose good region is narrow compared with the budget (a low pilot share) | it moves its draws away from where the rest of the trials were, trial by trial | round 2: about twice random's distinct `dinov2-small` runs per search, a best score 1.0 point more optimistic (+3.1 standard errors), and winners not distinguishable from random's on fresh seeds | a budget where its concentration pays in the model: E1 (c) |
| population | when a model of the space is not to be trusted, with a budget of many generations; a standard choice for several objectives (notebook 04) | it keeps good candidates and recombines them, builds no model, learns once per generation, and can repeat trials | both rounds: not distinguishable from random on either measure; 6 to 14 of its 40 trials per search were exact repeats; a space where a model misleads is not run here | many generations: E2 |

And one rule above all four: **read a comparison of samplers on fresh seeds, not on the
searches' best scores, and read "no result" with its resolution.** When the configurations
found do not differ there by more than the comparison can see, the sampler is not the thing
to change; a narrower space, or more seeds per configuration to tell good configurations
apart, is.

## 3.6 The number you may report

The winner of round 2 is the best of all its searches, each trial one run of 5 epochs, scored
on validation. Its validation score is the number the search *chose by*, and two things
separate it from a number you may report:

1. **Selection.** It is the maximum over hundreds of trials. The same configuration trained
   again with another seed scores, on the same split and for the same epochs, what that
   configuration scores, not what its luckiest run scored. §3.5 measured this optimism for
   every search; here it is followed for one winner all the way to the reported number.
2. **Split and length.** The search looked at validation only, for 5 epochs. The reported
   number comes from test, which the search never saw, at a length you can defend.

Each is measured on its own below. The winner is trained once more with its own search
seed, to show that the search's number is that one run, and then on the six `FRESH_SEEDS`
that no search used, for up to `FINAL_EPOCHS` epochs, recording validation and test accuracy
after every epoch. The training length is chosen first, on the validation curves alone;
test is read only after that choice is fixed.

**Before you run it, write down:** will the retrained winner, on validation after 5
epochs, score higher, equal or lower than its score in the search?

In [ ]:
everything = winners(round2)
top = everything.loc[everything["val"].idxmax()]
WINNER = {
    k: (v.item() if hasattr(v, "item") else v)
    for k, v in top[["encoder", "width", "depth", "lr"]].items()
}
# a run is a configuration and a torch seed: TPE's startup trials and the population's
# first generation are random's trials of the same seed, so they are the same runs
distinct = {
    (
        seed,
        *(t.user_attrs["config"][k] for k in ("encoder", "width", "depth")),
        round(t.params["lr"], 10),
    )
    for per_seed in round2.values()
    for seed, log in zip(SEEDS, per_seed)
    for t in log.trials(optuna.trial.TrialState.COMPLETE)
}
print(
    f"winner of all round-2 searches: {top['sampler']} seed {top['seed']}, "
    f"val {top['val']:.4f} after {SEARCH_EPOCHS} epochs; the best of {len(distinct)} distinct "
    f"runs among the {len(trials2)} trials of round 2"
)
print(
    "config:",
    {k: round(v, 5) if isinstance(v, float) else v for k, v in WINNER.items()},
)
own = train_head(WINNER, int(top["seed"]), SEARCH_EPOCHS)["val"][-1]
print(f"trained again with its search seed {top['seed']}: val {own:.4f}")

start = time.time()
retrain = [train_head(WINNER, s, FINAL_EPOCHS, on=("val", "test")) for s in FRESH_SEEDS]
print(
    f"retrained on {len(FRESH_SEEDS)} fresh seeds for {FINAL_EPOCHS} epochs in {time.time() - start:.0f}s"
)
val = np.array([r["val"] for r in retrain])  # (seeds, epochs)
test = np.array([r["test"] for r in retrain])  # not read until REPORT_EPOCHS is fixed

# the length, chosen on validation alone by the one-standard-error rule: the shortest
# training whose mean lies within one standard error (seed std / sqrt(seeds)) of the
# highest mean
mean_val = val.mean(0)
peak = int(mean_val.argmax())
se_peak = val[:, peak].std(ddof=1) / np.sqrt(len(FRESH_SEEDS))
REPORT_EPOCHS = int(np.argmax(mean_val >= mean_val[peak] - se_peak)) + 1
print(
    f"highest mean validation {mean_val[peak]:.4f} at epoch {peak + 1} of {FINAL_EPOCHS}, "
    f"standard error {se_peak:.4f}; epoch {REPORT_EPOCHS} is the first within it, at "
    f"{mean_val[REPORT_EPOCHS - 1]:.4f}"
)
print(f"REPORT_EPOCHS = {REPORT_EPOCHS}")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, FINAL_EPOCHS + 1)
ax.plot(x, val.mean(0), label="validation, mean over the fresh seeds")
ax.fill_between(
    x, val.mean(0) - val.std(0, ddof=1), val.mean(0) + val.std(0, ddof=1), alpha=0.2
)
ax.axvline(
    SEARCH_EPOCHS, color="k", ls=":", lw=1, label=f"the search's {SEARCH_EPOCHS} epochs"
)
ax.axvline(
    REPORT_EPOCHS,
    color="tab:green",
    ls="-.",
    lw=1,
    label=f"REPORT_EPOCHS = {REPORT_EPOCHS}",
)
ax.axhline(
    top["val"], color="crimson", ls="--", lw=1, label="the winner's score in the search"
)
ax.set_xlabel("epoch")
ax.set_ylabel("validation accuracy")
ax.set_title(
    f"the winner retrained on {len(FRESH_SEEDS)} fresh seeds: mean and std per epoch"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

**The search's number is one lucky run.** Trained again with its own search seed 0, the
winner gives 0.7846 again, to the last digit: the search's number is that one run. On the six
fresh seeds the same configuration, on the same split after 5 epochs, scores 0.768 on
average. The search kept the configuration *and* its lucky seed.

**The length**, read from validation alone: the mean climbs steeply for 4 to 5 epochs, then
creeps up slowly to its highest point, 0.7757 at epoch 29 of 30. That highest point is a
poor choice on its own: it is the maximum of 30 noisy means, so it is itself a little lucky,
and it sits at the edge of the window, where a longer window might move it again. The rule
used here is the one-standard-error rule: take the standard error of the highest mean over
the six seeds (0.0030), and pick the shortest training whose mean lies within one standard
error of it. Epoch 12 is the first, at 0.7733, so `REPORT_EPOCHS = 12`. The rule takes the
cheaper training whenever its mean lies within the noise of the best one. Only now is test read.

The steps from the search's number to the one you report, one change at a time:

In [ ]:
def at(curves: np.ndarray, epochs: int) -> str:
    """Mean and std over the seeds after `epochs` epochs."""
    column = curves[:, epochs - 1]
    return f"{column.mean():.4f}, seed std {column.std(ddof=1):.4f}"


rows = [
    (
        f"search: the best trial, one seed, val, {SEARCH_EPOCHS} epochs",
        f"{top['val']:.4f}",
    ),
    (f"retrained: fresh seeds, val, {SEARCH_EPOCHS} epochs", at(val, SEARCH_EPOCHS)),
    (f"retrained: fresh seeds, val, {REPORT_EPOCHS} epochs", at(val, REPORT_EPOCHS)),
    (f"retrained: fresh seeds, test, {REPORT_EPOCHS} epochs", at(test, REPORT_EPOCHS)),
]
print(f"{'':48s} accuracy")
for label, number in rows:
    print(f"{label:48s} {number}")
diff, se, ratio = paired(val[:, REPORT_EPOCHS - 1], val[:, SEARCH_EPOCHS - 1])
print(
    f"\nlength, {SEARCH_EPOCHS} to {REPORT_EPOCHS} epochs on validation: {diff:+.4f}, "
    f"standard error {se:.4f}, {ratio:+.1f} standard errors (paired by seed)"
)


def sampling_se(accuracy: float, n: int) -> float:
    """The binomial standard error of an accuracy measured on n images: every image is
    right with chance `accuracy`, so the share right moves by about this much when the n
    images are drawn again."""
    return float(np.sqrt(accuracy * (1 - accuracy) / n))


n_val, n_test = (
    len(SPLITS["dinov2-small"]["val"][1]),
    len(SPLITS["dinov2-small"]["test"][1]),
)
p_val, p_test = val[:, REPORT_EPOCHS - 1].mean(), test[:, REPORT_EPOCHS - 1].mean()
se_val, se_test = sampling_se(p_val, n_val), sampling_se(p_test, n_test)
print(f"standard error from the {n_test} test images alone: {se_test:.4f}")
print(
    f"validation against test: {p_test - p_val:+.4f}, standard error of that difference "
    f"(two sets of images) {np.sqrt(se_val**2 + se_test**2):.4f}"
)

runs = StudyLog.runs(
    name="lesson3-algorithm-runs", choices={"encoder": ENCODERS, "dataset": ["dtd"]}
)
final_trial = runs.log(
    name="round-2 winner, retrained",
    role="final",
    results=Scalars(accuracy=list(test[:, REPORT_EPOCHS - 1])),
    dataset="dtd",
    params={"dataset": "dtd", **WINNER, "epochs": REPORT_EPOCHS},
    attrs={"split": "test", "found_by": str(top["sampler"])},
)

Read the table one row at a time; each row changes one thing:

1. **Selection** (row 1 to row 2): 0.785 to 0.768, 1.7 points, all of it from keeping the
   best of 790 distinct runs (the 960 trials of round 2 include TPE's and the population's
   first trials, which are random's runs of the same seed, and the population's repeats).
   This is the optimism of a search winner, and it is the largest step in the table. Row 2 is
   still not free of selection: it is the configuration that *looked* best on validation, and
   only a split that played no part in choosing it can show how much of its 0.768 is the
   configuration and how much is the choice.
2. **Length** (row 2 to row 3, validation, 5 to 12 epochs): 0.768 to 0.773, +0.55 points at
   +4.8 standard errors, paired by seed: a result, but a small one. Training longer than the
   search did buys a little on this head.
3. **Split** (row 3 to row 4, validation to test, 12 epochs): 0.773 to 0.782. The seeds say
   nothing about this step, because they share the same 752 test images. What moves an
   accuracy from one set of images to another is which images are in it. Each image is
   either right or wrong, so the share right is a binomial count, and its standard error is
   $\sqrt{p(1-p)/n}$: for $p \approx 0.79$ on $n = 752$ images, 0.015. The difference between
   two such sets has a standard error of 0.021, and the +0.9 points lie well inside it.

**The number to report is 0.782 on test, the winner retrained for 12 epochs over six fresh
seeds: seed std 0.004, and a standard error of 0.015 from the 752 test images.** The second
is the larger by far, and it is the one to give with the number: another 752 images would
move the score by about that much, whatever the seeds. No search and no choice in this
notebook looked at test. The run is written to the study `lesson3-algorithm-runs` with role
`final`.

That the test score here lies *above* validation is a property of these 752 rows, not a
rule: within 0.021 the order of the two can come out either way. What is a rule is that the
search's own number lies above what its winner gives again, because it was chosen as a
maximum.

One caution for using 0.782 outside this notebook: DINOv2's pretraining data was assembled
by retrieving web images similar to public benchmark datasets, and DTD's training split is
among the datasets used for that retrieval (DINOv2 paper, Table 15). This notebook's rows
mix DTD's own splits, so some test rows may be close to, or among, what the encoder saw in
pretraining. That is no leak into the head, which never trains on a test row, but it makes
0.782 optimistic as a measure of textures the encoder has never seen. The comparisons
between samplers and rounds do not depend on it.

## What you now know

- **What each sampler knows.** Grid knows a lattice and its cost is the product of the axes
  (45 points for the head, 135 with the encoder). Random draws every axis afresh and gives
  every trial a new value (240 learning rates against grid's 3), but never learns what it
  has seen. TPE splits the trials so far into a good tenth and the rest, models each group
  with one kernel per trial over all axes together, and runs the candidate most typical of
  the good group and least of the rest, after 10 random trials. A population keeps the best
  8 trials and makes children from them by crossover and mutation; it builds no model,
  learns between generations, and can repeat a trial it already ran.
- **Compare samplers on fresh seeds, not on their best scores.** A search's best score is the
  maximum of its noisy runs. In round 2 TPE's best score was 1.1 points above random's
  (+3.2 standard errors), but on fresh seeds the configurations they found gave no result
  (+0.3 standard errors). "No result" is not "the same": at six seeds the comparison rules
  out an advantage of more than about 0.6 points, not a smaller one. TPE ran 26 to 27
  distinct `dinov2-small` runs per search against random's 10 to 16, and the best of more
  runs lies higher. On this space, at this budget, the samplers differ in where they spend
  their trials and in how optimistic their best score is; in the model they find, the
  fresh-seed comparison shows no difference larger than about 0.6 points.
- **No result has a reason.** On the head the trials ran from 0.10 to 0.785, so the space is not
  flat, yet the 24 winners lie between 0.752 and 0.772 on fresh seeds: every sampler reached
  the wide good region, and none picked out a configuration measurably better inside it.
- **What an axis costs.** Adding two encoders that are fifteen points worse took about two
  thirds of random's and grid's trials, and cost random 1.1 points of best score (−4.0
  standard errors), but its winners on fresh seeds lost nothing the line can see (−1.2).
- **Choose a sampler for the space and the budget, and check the choice on fresh seeds.**
  Grid when the lattice fits the budget and holds the values you need; random for a small
  budget or no prior knowledge, and as the control; TPE when the budget is well past its
  startup and the good region is narrow compared with the budget, which a small random pilot
  shows; a population when a model of the space is not to be trusted, with many
  generations, and as a standard choice for several objectives (04). Where the
  configurations found cannot be told apart, change the space or add seeds, not the
  sampler.
- **Read the shape of the best-so-far curve, as a curve of the best score.** Flat early:
  change the space. Still climbing: more trials would raise the best score; whether they
  would find a better model, fresh seeds tell.
- **Report the test number of a retrained winner, with both uncertainties.** The search's
  0.785 was its best of 790 distinct runs; the same configuration on six fresh seeds gives
  0.768 on validation at 5 epochs, and 0.782 on test at 12 epochs, the shortest training
  within one standard error of the best on validation. Selection was the largest step. Give
  the test number with the standard error of its 752 images (0.015), not only the seed std
  (0.004).

| sampler | what it knows | removes this shortcoming of the one before | where it fell short here |
|---|---|---|---|
| grid | a lattice per axis | | 3 learning rates only; spends evenly on bad encoders |
| random | the distribution per axis | grid's lattice: a new value of every axis in every trial | spends two thirds of round 2 on bad encoders (its winners did not suffer on fresh seeds) |
| TPE | which combinations of values the good trials had, against the rest | random's blindness to the trials so far | its concentration raised its best score; its winners were not distinguishable from random's on fresh seeds |
| population (NSGA-II) | the best 8 trials so far, recombined; no model | TPE's reliance on its model: it keeps the good candidates themselves and recombines them | repeats trials (6 to 14 of 40) and learns per generation; not distinguishable from random here; a space where a model misleads is not run |

Notebook 04 takes the budget question further: stop the trials that are clearly losing
after a few epochs (a pruner), and read the best trial against a second axis, the number of
parameters.

## Exercises

**E1: choose before you run.** For each space and budget below, write down which sampler you
would use and why, in terms of what it knows and of a run in this notebook. Only then run
all four with `race(<space>, budget=<budget>)`, and read the result in standard errors on
both measures, `measures(...)` and `versus_random(...)`: does the sampler you chose find a
better configuration on fresh seeds, or only a higher best score?

- (a) The head of round 1 with `lr` fixed at `1e-3`: width and depth only, 15
  combinations, a budget of 15. (Add the space to `SPACE`; grid takes its lattice for the axes the space searches.)
- (b) The space of round 2 with a budget of 10.
- (c) The space of round 2 with a budget of 120.

Where the result goes against your choice, say which of your reasons it refutes.

**E2: give the population room.** Run round 2 again with `budget=120` and
`population_size=20` for the population (six generations). Keep random, grid and TPE at the
same budget: equal trials is the control. Count its repeats again. Does the population
catch up with TPE on fresh seeds?

**E3: how much budget did the encoder axis cost random?** Count, per random search of
round 2, the trials that drew `dinov2-small`, and compare its best-so-far curve on those
trials only with round 1's random curve at the same number of `dinov2-small` trials. If the
two lie on each other, what does that say about where round 2's loss came from?

**E4: a space where configurations differ more.** Round 1's samplers could not be told
apart because the configurations they found lay close together. Train on 10%
of the training rows and search over regularisation (weight decay, Adam's `weight_decay`
argument, which pulls every weight a little towards zero at each step; or the
`GaussianNoise` sigma of lesson 1). Write down first whether you expect the samplers to
separate now, and why; then read the race on fresh seeds, as in §3.5.

**E5: TPE's start and its model.** Set `n_startup_trials=20` and then `5` in `TPESampler`.
What changes in round 2, in the number of trials on the weak encoders and in the best score?
Then set `multivariate=False`: does a TPE that models every axis on its own do worse here,
and would you expect it to on a space where only combinations score well?